# Sumarização de Textos com PLN: Similaridade do Cosseno e PageRank

**Autor: Wellington M Santos - Data Scientist**

[![LinkedIn](https://img.shields.io/badge/LinkedIn-wellington--moreira--santos-blue)](https://www.linkedin.com/in/wellington-moreira-santos)
[![Email](https://img.shields.io/badge/Email-wsantos08%40hotmail.com-red)](mailto:wsantos08@hotmail.com)


---

## 1. Introdução

Nos dois projetos anteriores, cada sentença era avaliada de forma independente. No [algoritmo de frequência simples](Algoritmo%20baseado%20em%20frequencia.ipynb), somei os pesos das palavras importantes que a sentença continha. No [algoritmo de Luhn](Algoritmo%20de%20Luhn.ipynb), avaliei a densidade dos clusters de palavras importantes dentro dela. Em ambos os casos, a sentença era uma unidade isolada, pontuada por seu próprio conteúdo.

Neste projeto, a lógica muda. Em vez de avaliar cada sentença sozinha, avalio o quanto ela se parece com todas as outras sentenças do texto. A ideia é que uma sentença verdadeiramente central para o conteúdo tende a ser similar a muitas outras sentenças, porque compartilha vocabulário e ideias com diferentes partes do texto. Sentença periférica, por outro lado, fala de algo específico que não ressoa em nenhuma outra parte.

Para medir essa similaridade, uso a similaridade do cosseno: represento cada par de sentenças como vetores no espaço das palavras e meço o ângulo entre eles. Quanto menor o ângulo, mais similares as sentenças. Com essas similaridades calculadas para todos os pares, construo um grafo onde cada nó é uma sentença e cada aresta tem o peso da similaridade entre elas. Aplico então o PageRank sobre esse grafo para identificar as sentenças mais centrais na rede de significado do texto.

Essa abordagem é conhecida na literatura como TextRank, proposta por Mihalcea e Tarau em 2004, e é a base de algoritmos de sumarização ainda usados em produção hoje.

**Stack utilizada:** `Python 3.10+`, `nltk`, `numpy`, `scipy`, `networkx`, `re`, `string`, `newspaper4k`, `spacy>=3.x`, `rouge-score`, `IPython.display`

**Referências:** [Cosine Similarity — Wikipedia](https://en.wikipedia.org/wiki/Cosine_similarity) | [PageRank — Wikipedia](https://en.wikipedia.org/wiki/PageRank)

**Seções:**

1. Introdução
2. Instalação e Configuração
3. Pré-processamento do Texto
4. Similaridade do Cosseno
5. Matriz de Similaridade
6. PageRank e Geração do Resumo
7. Visualização do Resumo
8. Extração de Texto da Web
9. Extensão: Lematização com spaCy
10. Avaliação com ROUGE: Três Algoritmos Comparados
11. Conclusão e Considerações Finais



---



## 2. Instalação e Configuração


In [ ]:
# Instalar dependências (executar uma vez)
# !pip install nltk numpy scipy networkx newspaper4k spacy rouge-score
# !python -m spacy download pt_core_news_sm

In [1]:
import re
import string

import nltk
import numpy as np
from scipy.spatial.distance import cosine as cosine_distance
import networkx as nx

from IPython.display import HTML, display

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\wsant\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\wsant\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\wsant\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True


---



## 3. Pré-processamento do Texto

O pré-processamento segue o mesmo padrão estabelecido nos projetos anteriores: lowercase, tokenização, remoção de stopwords e pontuação. Aqui adiciono "ser" e "além" à lista de stopwords customizadas, como no projeto de Luhn, porque aparecem com frequência no texto de exemplo sem carregar carga temática.


In [2]:
stopwords = nltk.corpus.stopwords.words('portuguese')
stopwords.extend(['ser', 'além'])
print(f"Total de stopwords: {len(stopwords)}")

Total de stopwords: 209


In [3]:
def preprocessamento(texto):
    """
    Normaliza o texto: lowercase, tokenização, remoção de stopwords e pontuação.

    Parâmetros:
        texto (str): texto bruto de entrada

    Retorna:
        str: string com tokens relevantes separados por espaço
    """
    texto_formatado = texto.lower()
    tokens = nltk.word_tokenize(texto_formatado, language='portuguese')
    tokens = [
        palavra for palavra in tokens
        if palavra not in stopwords and palavra not in string.punctuation
    ]
    return ' '.join([t for t in tokens if not t.isdigit()])

In [4]:
texto_original = """A inteligência artificial é a inteligência similar à humana máquinas.
                    Definem como o estudo de agente artificial com inteligência.
                    Ciência e engenharia de produzir máquinas com inteligência.
                    Resolver problemas e possuir inteligência.
                    Relacionada ao comportamento inteligente.
                    Construção de máquinas para raciocinar.
                    Aprender com os erros e acertos.
                    Inteligência artificial é raciocinar nas situações do cotidiano."""

texto_original = re.sub(r'\s+', ' ', texto_original)

In [5]:

sentencas_originais = nltk.sent_tokenize(texto_original, language='portuguese')
sentencas_formatadas = [preprocessamento(s) for s in sentencas_originais]

print("Sentenças originais:")
for i, s in enumerate(sentencas_originais, 1):
    print(f"  [{i}] {s}")

print("\nSentenças pré-processadas:")
for i, s in enumerate(sentencas_formatadas, 1):
    print(f"  [{i}] {s}")

Sentenças originais:
  [1] A inteligência artificial é a inteligência similar à humana máquinas.
  [2] Definem como o estudo de agente artificial com inteligência.
  [3] Ciência e engenharia de produzir máquinas com inteligência.
  [4] Resolver problemas e possuir inteligência.
  [5] Relacionada ao comportamento inteligente.
  [6] Construção de máquinas para raciocinar.
  [7] Aprender com os erros e acertos.
  [8] Inteligência artificial é raciocinar nas situações do cotidiano.

Sentenças pré-processadas:
  [1] inteligência artificial inteligência similar humana máquinas
  [2] definem estudo agente artificial inteligência
  [3] ciência engenharia produzir máquinas inteligência
  [4] resolver problemas possuir inteligência
  [5] relacionada comportamento inteligente
  [6] construção máquinas raciocinar
  [7] aprender erros acertos
  [8] inteligência artificial raciocinar situações cotidiano



---



## 4. Similaridade do Cosseno

Antes de medir a similaridade entre duas sentenças, preciso representá-las de forma que um algoritmo matemático consiga comparar. Faço isso construindo vetores de frequência de palavras.

O processo é simples: uno o vocabulário das duas sentenças em uma lista sem repetições e crio um vetor para cada sentença onde cada posição corresponde a uma palavra desse vocabulário conjunto. O valor em cada posição é quantas vezes aquela palavra aparece na sentença. Com os dois vetores construídos, calculo o cosseno do ângulo entre eles.

Um cosseno de 1 significa que os vetores apontam na mesma direção: as sentenças compartilham exatamente o mesmo vocabulário nas mesmas proporções. Um cosseno de 0 significa que os vetores são perpendiculares: as sentenças não têm nenhuma palavra em comum.


In [6]:
def calcula_similaridade_sentencas(sentenca1, sentenca2):
    """
    Calcula a similaridade do cosseno entre duas sentenças.
    Representa cada sentença como um vetor de frequência de palavras
    no espaço do vocabulário conjunto das duas sentenças.

    Parâmetros:
        sentenca1 (str): primeira sentença pré-processada
        sentenca2 (str): segunda sentença pré-processada

    Retorna:
        float: similaridade do cosseno entre 0 e 1
    """
    palavras1 = nltk.word_tokenize(sentenca1)
    palavras2 = nltk.word_tokenize(sentenca2)

    vocabulario = list(set(palavras1 + palavras2))

    vetor1 = [palavras1.count(p) for p in vocabulario]
    vetor2 = [palavras2.count(p) for p in vocabulario]

    return 1 - cosine_distance(vetor1, vetor2)


In [7]:
# Similaridade de uma sentença consigo mesma deve ser 1.0
print(calcula_similaridade_sentencas(sentencas_formatadas[0], sentencas_formatadas[0]))

# Similaridade entre sentenças distintas
print(calcula_similaridade_sentencas(sentencas_formatadas[0], sentencas_formatadas[1]))
print(calcula_similaridade_sentencas(sentencas_formatadas[0], sentencas_formatadas[-1]))

1.0
0.4743416490252569
0.4743416490252569



---



## 5. Matriz de Similaridade

Com a função de similaridade definida, calculo a similaridade entre todos os pares de sentenças do texto e armazeno em uma matriz. A posição `[i][j]` da matriz contém a similaridade entre a sentença `i` e a sentença `j`. A diagonal principal fica zerada: não faz sentido conectar uma sentença a ela mesma no grafo que vem a seguir.


In [8]:
def calcula_matriz_similaridade(sentencas):
    """
    Calcula a matriz de similaridade do cosseno para um conjunto de sentenças.
    A diagonal principal é mantida em zero para evitar auto-conexões no grafo.

    Parâmetros:
        sentencas (list[str]): sentenças pré-processadas

    Retorna:
        np.ndarray: matriz quadrada de similaridades de shape (n, n)
    """
    n = len(sentencas)
    matriz = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            if i != j:
                matriz[i][j] = calcula_similaridade_sentencas(sentencas[i], sentencas[j])

    return matriz


In [9]:
matriz = calcula_matriz_similaridade(sentencas_formatadas)
print(f"Shape da matriz: {matriz.shape}")
print(matriz.round(3))

Shape da matriz: (8, 8)
[[0.    0.474 0.474 0.354 0.    0.204 0.    0.474]
 [0.474 0.    0.2   0.224 0.    0.    0.    0.4  ]
 [0.474 0.2   0.    0.224 0.    0.258 0.    0.2  ]
 [0.354 0.224 0.224 0.    0.    0.    0.    0.224]
 [0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.204 0.    0.258 0.    0.    0.    0.    0.258]
 [0.    0.    0.    0.    0.    0.    0.    0.   ]
 [0.474 0.4   0.2   0.224 0.    0.258 0.    0.   ]]



Cada linha representa uma sentença e seus valores de similaridade com todas as outras. Uma linha com valores altos em muitas colunas indica que a sentença é similar a muitas outras, ou seja, que ela é central para o conteúdo do texto.



---



## 6. PageRank e Geração do Resumo

A matriz de similaridade é a representação numérica de um grafo: cada sentença é um nó, e o peso de cada aresta é a similaridade entre o par de sentenças conectado. Converto essa matriz em um grafo com `networkx` e aplico o PageRank.

O PageRank, originalmente o algoritmo que o Google usava para rankear páginas web, funciona com uma analogia de navegação aleatória: imagine um usuário que começa em um nó aleatório do grafo e a cada passo salta para um nó vizinho com probabilidade proporcional ao peso da aresta. A pontuação de cada nó é proporcional à probabilidade de o usuário estar nele após muitos passos. Nós com muitas conexões de peso alto recebem mais visitas e pontuam mais. No contexto de sumarização, a "visita" equivale a relevância: uma sentença muito similar a muitas outras é mais provável de ser visitada e recebe pontuação mais alta.


In [10]:
def sumarizar(texto, quantidade_sentencas):
    """
    Sumariza um texto usando similaridade do cosseno e PageRank.

    Parâmetros:
        texto (str): texto original a ser sumarizado
        quantidade_sentencas (int): número de sentenças no resumo

    Retorna:
        tuple: (sentencas_originais, melhores_sentencas, notas_ordenadas)
    """
    sentencas_originais = nltk.sent_tokenize(texto, language='portuguese')
    sentencas_formatadas = [preprocessamento(s) for s in sentencas_originais]

    matriz_similaridade = calcula_matriz_similaridade(sentencas_formatadas)
    grafo = nx.from_numpy_array(matriz_similaridade)
    notas = nx.pagerank(grafo)

    notas_ordenadas = sorted(
        ((notas[i], sentenca) for i, sentenca in enumerate(sentencas_originais)),
        reverse=True
    )

    melhores_sentencas = [sentenca for _, sentenca in notas_ordenadas[:quantidade_sentencas]]

    return sentencas_originais, melhores_sentencas, notas_ordenadas

In [11]:

sentencas_originais, melhores_sentencas, notas_sentencas = sumarizar(texto_original, 3)

print("Notas por sentença (PageRank):")
for nota, sentenca in notas_sentencas:
    marcador = ">>" if sentenca in melhores_sentencas else "  "
    print(f"  {marcador} [{nota:.4f}] {sentenca}")

Notas por sentença (PageRank):
  >> [0.2282] A inteligência artificial é a inteligência similar à humana máquinas.
  >> [0.1839] Inteligência artificial é raciocinar nas situações do cotidiano.
  >> [0.1633] Ciência e engenharia de produzir máquinas com inteligência.
     [0.1544] Definem como o estudo de agente artificial com inteligência.
     [0.1264] Resolver problemas e possuir inteligência.
     [0.0962] Construção de máquinas para raciocinar.
     [0.0238] Relacionada ao comportamento inteligente.
     [0.0238] Aprender com os erros e acertos.



A construção de `melhores_sentencas` usa um slice direto `[:quantidade_sentencas]` sobre a lista já ordenada, em vez do loop `for i in range(quantidade_sentencas)` do script original. Mais conciso, mesmo resultado.



---



## 7. Visualização do Resumo

Mantenho a função de visualização com detecção de ambiente estabelecida nos projetos anteriores.


In [12]:
def visualiza_resumo(titulo, lista_sentencas, melhores_sentencas):
    """
    Exibe o texto original com as sentenças do resumo destacadas.
    Suporta Jupyter (HTML) e outros ambientes (texto puro).

    Parâmetros:
        titulo (str): título exibido no cabeçalho
        lista_sentencas (list): todas as sentenças do texto original
        melhores_sentencas (list): sentenças selecionadas para o resumo
    """
    try:
        get_ipython  # noqa
        texto_html = ''
        for sentenca in lista_sentencas:
            if sentenca in melhores_sentencas:
                texto_html += f'<mark>{sentenca}</mark> '
            else:
                texto_html += sentenca + ' '
        display(HTML(f'<h3>Resumo: {titulo}</h3><p>{texto_html}</p>'))
    except NameError:
        print(f"\n=== Resumo: {titulo} ===")
        for sentenca in lista_sentencas:
            marcador = ">> " if sentenca in melhores_sentencas else "   "
            print(f"{marcador}{sentenca}")


In [13]:

visualiza_resumo('Texto de Exemplo', sentencas_originais, melhores_sentencas)


---



## 8. Extração de Texto da Web


In [14]:

# !pip install newspaper4k
from newspaper import Article


In [15]:
def extrair_artigo(url):
    """
    Extrai título e texto principal de uma URL usando newspaper4k.

    Parâmetros:
        url (str): endereço do artigo

    Retorna:
        tuple: (titulo, texto) ou (None, None) em caso de erro
    """
    try:
        artigo = Article(url, language='pt')
        artigo.download()
        artigo.parse()
        return artigo.title, artigo.text
    except Exception as e:
        print(f"Erro ao extrair {url}: {e}")
        return None, None


In [16]:
url = 'https://agenciabrasil.ebc.com.br/economia/noticia/2024-01/fmi-inteligencia-artificial-afetara-40-dos-empregos-em-todo-o-mundo'
titulo, texto = extrair_artigo(url)

print(f"Título: {titulo}")
print(f"Tamanho do texto: {len(texto)} caracteres")

Título: FMI: inteligência artificial afetará 40% dos empregos em todo o mundo
Tamanho do texto: 2124 caracteres


In [17]:
sentencas_originais, melhores_sentencas, notas_sentencas = sumarizar(texto, 5)

print("Sentenças selecionadas:")
for s in melhores_sentencas:
    print(f"  - {s}")


Sentenças selecionadas:
  - "É certo que haverá impacto" disse Georgieva, observando que a IA pode acabar com alguns empregos e melhorar outros.
  - "No mundo, 40% dos empregos serão afetados.
  - "A IA pode ser assustadora, mas também pode ser uma grande oportunidade para todos", observou.
  - A diretora defendeu que a prioridade deve ser ajudar os trabalhadores afetados e "partilhar os ganhos de produtividade".
  - "Devemos concentrar-nos nos países de rendimento mais baixo", destacou a diretora-geral do FMI, que demonstrou receio com o risco de abandono escolar nos Estados mais pobres.


In [18]:
visualiza_resumo(titulo, sentencas_originais, melhores_sentencas)


Uma observação importante sobre performance: a complexidade desta abordagem é O(n²) no número de sentenças, porque calculo a similaridade para cada par possível. Em artigos curtos isso é imperceptível, mas em textos muito longos (centenas de sentenças) o tempo de execução cresce rapidamente. Para esses casos, estratégias como janela deslizante ou amostragem de pares são alternativas práticas.



---



## 9. Extensão: Lematização com spaCy


In [19]:
# !python -m spacy download pt_core_news_sm
import spacy

pln = spacy.load('pt_core_news_sm')
pln

In [20]:
def preprocessamento_lematizacao(texto):
    """
    Variante do pré-processamento com lematização via spaCy 3.x.

    Parâmetros:
        texto (str): texto bruto de entrada

    Retorna:
        str: string com lemas relevantes separados por espaço
    """
    texto = re.sub(r'\s+', ' ', texto.lower())
    documento = pln(texto)
    tokens = [
        token.lemma_ for token in documento
        if token.lemma_ not in stopwords
        and token.lemma_ not in string.punctuation
        and not token.is_digit
        and not token.is_space
    ]
    return ' '.join(tokens)

In [21]:
def sumarizar_lematizacao(texto, quantidade_sentencas):
    """
    Pipeline de sumarização com similaridade do cosseno e PageRank,
    usando lematização no pré-processamento.

    Parâmetros:
        texto (str): texto original a ser sumarizado
        quantidade_sentencas (int): número de sentenças no resumo

    Retorna:
        tuple: (sentencas_originais, melhores_sentencas, notas_ordenadas)
    """
    sentencas_originais = nltk.sent_tokenize(texto, language='portuguese')
    sentencas_formatadas = [preprocessamento_lematizacao(s) for s in sentencas_originais]

    matriz_similaridade = calcula_matriz_similaridade(sentencas_formatadas)
    grafo = nx.from_numpy_array(matriz_similaridade)
    notas = nx.pagerank(grafo)

    notas_ordenadas = sorted(
        ((notas[i], sentenca) for i, sentenca in enumerate(sentencas_originais)),
        reverse=True
    )

    melhores_sentencas = [sentenca for _, sentenca in notas_ordenadas[:quantidade_sentencas]]

    return sentencas_originais, melhores_sentencas, notas_ordenadas

In [22]:
sentencas_originais, melhores_sentencas, _ = sumarizar(texto, 5)
visualiza_resumo(f"{titulo} (tokenização simples)", sentencas_originais, melhores_sentencas)

In [23]:
sentencas_originais, melhores_sentencas, _ = sumarizar_lematizacao(texto, 5)
visualiza_resumo(f"{titulo} (lematização)", sentencas_originais, melhores_sentencas)


---



## 10. Avaliação com ROUGE: Três Algoritmos Comparados

Este é o terceiro projeto da série de sumarização. Faz sentido encerrar com uma comparação completa: frequência simples, Luhn e similaridade do cosseno, todos avaliados no mesmo artigo e contra o mesmo resumo de referência.


In [24]:

# !pip install rouge-score
from rouge_score import rouge_scorer

In [25]:
def avaliar_rouge(resumo_gerado, resumo_referencia):
    """
    Calcula métricas ROUGE entre o resumo gerado e o de referência.

    Parâmetros:
        resumo_gerado (str): resumo produzido pelo algoritmo
        resumo_referencia (str): resumo de referência escrito por humano

    Retorna:
        dict: scores ROUGE-1, ROUGE-2 e ROUGE-L
    """
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    return scorer.score(resumo_referencia.strip(), resumo_gerado.strip())


In [26]:

# Resumo de referência escrito manualmente
# Fonte: https://agenciabrasil.ebc.com.br/economia/noticia/2024-01/fmi-inteligencia-artificial-afetara-40-dos-empregos-em-todo-o-mundo
resumo_referencia = """
O FMI alerta que a inteligência artificial afetará 40% dos empregos em todo o mundo,
com impacto ainda maior nas economias avançadas, onde 60% dos postos de trabalho
serão impactados. A diretora-geral do fundo ressalta que os efeitos não são
necessariamente negativos e podem resultar em aumento de rendimentos, mas alerta
para o risco de aprofundamento das desigualdades entre países com diferentes
capacidades de adaptação tecnológica.
"""


In [27]:

from collections import Counter
import heapq

def sumarizar_frequencia_simples(texto, quantidade_sentencas):
    """Algoritmo de frequência simples para comparação."""
    texto_formatado = preprocessamento(texto)
    contagem = Counter(nltk.word_tokenize(texto_formatado))
    freq_max = max(contagem.values())
    pesos = {p: f / freq_max for p, f in contagem.items()}

    sentencas = nltk.sent_tokenize(texto, language='portuguese')
    notas = {}
    for sentenca in sentencas:
        for palavra in nltk.word_tokenize(sentenca.lower(), language='portuguese'):
            if palavra in pesos:
                notas[sentenca] = notas.get(sentenca, 0) + pesos[palavra]

    return heapq.nlargest(quantidade_sentencas, notas, key=notas.get)



In [28]:
def sumarizar_luhn(texto, top_n_palavras, distancia, quantidade_sentencas):
    """Algoritmo de Luhn para comparação."""
    sentencas_originais = nltk.sent_tokenize(texto, language='portuguese')
    sentencas_formatadas = [preprocessamento(s) for s in sentencas_originais]

    palavras = [p for s in sentencas_formatadas for p in nltk.word_tokenize(s)]
    frequencia = Counter(palavras)
    top_palavras = [p for p, _ in frequencia.most_common(top_n_palavras)]

    def calcula_notas_luhn(sentencas, importantes, distancia):
        notas = []
        for idx, sentenca in enumerate(
            [nltk.word_tokenize(s.lower(), language='portuguese') for s in sentencas]
        ):
            indices = sorted([sentenca.index(p) for p in importantes if p in sentenca])
            if not indices:
                continue
            grupos = []
            grupo = [indices[0]]
            for i in range(1, len(indices)):
                if indices[i] - indices[i - 1] < distancia:
                    grupo.append(indices[i])
                else:
                    grupos.append(grupo)
                    grupo = [indices[i]]
            grupos.append(grupo)
            nota_max = max(len(g) ** 2 / (g[-1] - g[0] + 1) for g in grupos)
            notas.append((nota_max, idx))
        return notas

    notas = calcula_notas_luhn(sentencas_formatadas, top_palavras, distancia)
    melhores = heapq.nlargest(quantidade_sentencas, notas)
    return [sentencas_originais[i] for (_, i) in melhores]


In [29]:
# Gera resumos com os três algoritmos
melhores_freq = sumarizar_frequencia_simples(texto, 5)
melhores_luhn = sumarizar_luhn(texto, top_n_palavras=20, distancia=5, quantidade_sentencas=5)
_, melhores_cosseno, _ = sumarizar(texto, 5)

resumo_freq    = ' '.join(melhores_freq)
resumo_luhn    = ' '.join(melhores_luhn)
resumo_cosseno = ' '.join(melhores_cosseno)


In [30]:
scores_freq    = avaliar_rouge(resumo_freq, resumo_referencia)
scores_luhn    = avaliar_rouge(resumo_luhn, resumo_referencia)
scores_cosseno = avaliar_rouge(resumo_cosseno, resumo_referencia)

algoritmos = {
    'Frequência': scores_freq,
    'Luhn':       scores_luhn,
    'Cosseno':    scores_cosseno,
}

print("Comparação: Frequência vs. Luhn vs. Cosseno")
print("-" * 60)
print(f"{'Métrica':10} {'Frequência F1':>16} {'Luhn F1':>12} {'Cosseno F1':>13}")
print("-" * 60)
for metrica in ['rouge1', 'rouge2', 'rougeL']:
    scores = {nome: algoritmos[nome][metrica].fmeasure for nome in algoritmos}
    melhor = max(scores, key=scores.get)
    print(
        f"{metrica.upper():10} "
        f"{scores['Frequência']:>16.3f} "
        f"{scores['Luhn']:>12.3f} "
        f"{scores['Cosseno']:>13.3f}"
        f"   melhor: {melhor}"
    )


Comparação: Frequência vs. Luhn vs. Cosseno
------------------------------------------------------------
Métrica       Frequência F1      Luhn F1    Cosseno F1
------------------------------------------------------------
ROUGE1                0.318        0.430         0.341   melhor: Luhn
ROUGE2                0.080        0.224         0.123   melhor: Luhn
ROUGEL                0.179        0.291         0.207   melhor: Luhn



Os resultados obtidos sobre o artigo do FMI mostram um padrão consistente:

| Métrica  | Frequência F1 | Luhn F1 | Cosseno F1 |
|----------|---------------|---------|------------|
| ROUGE-1  | 0.318         | 0.430   | 0.341      |
| ROUGE-2  | 0.080         | 0.224   | 0.123      |
| ROUGE-L  | 0.179         | 0.291   | 0.207      |

O Luhn venceu nas três métricas. O que vale observar além do vencedor é a ordem: o cosseno ficou consistentemente entre os dois, superando a frequência simples mas sem alcançar o Luhn. Esse padrão faz sentido para este texto em particular. O artigo do FMI tem vocabulário temático concentrado e bem definido, exatamente a estrutura que o critério de proximidade do Luhn foi projetado para explorar. O cosseno, por avaliar a centralidade de cada sentença no grafo do texto inteiro, distribui a relevância de forma mais uniforme entre as sentenças, o que tende a ser uma vantagem em textos com múltiplos subtemas mas não necessariamente em artigos com foco temático único. A frequência simples, sem nenhum critério estrutural além da contagem bruta de palavras, fica em último nas três métricas. Vale reforçar que esses valores são específicos para este artigo e para os hiperparâmetros utilizados, e que a ordem entre os algoritmos pode se inverter dependendo da estrutura do texto avaliado.



---


## 11. Conclusão e Considerações Finais

### 11.1 Síntese do Projeto

Neste projeto implementei sumarização extrativa usando similaridade do cosseno e PageRank, a abordagem conhecida na literatura como TextRank. Cada sentença do texto é representada como um vetor de frequência de palavras; a similaridade do cosseno mede o ângulo entre pares de vetores; os valores formam uma matriz que se converte em grafo ponderado; e o PageRank identifica as sentenças mais centrais nesse grafo.

O pipeline inclui extração de artigos com `newspaper4k`, uma variante com lematização via spaCy 3.x, e uma avaliação comparativa com ROUGE que reúne os três algoritmos desenvolvidos ao longo da série. Na comparação sobre o artigo do FMI, o Luhn obteve os melhores resultados (ROUGE-1: 0.430, ROUGE-2: 0.224, ROUGE-L: 0.291), seguido pelo cosseno (0.341, 0.123, 0.207) e pela frequência simples (0.318, 0.080, 0.179), com o cosseno ficando consistentemente entre os dois em todas as métricas.

### 11.2 Limitações

A representação por vetores de frequência de palavras ignora a ordem e a semântica. Duas sentenças podem ser muito similares em significado mas usar vocabulários completamente diferentes, e o algoritmo as trataria como não relacionadas. O passo inverso também ocorre: sentenças que repetem as mesmas palavras em contextos distintos recebem alta similaridade mesmo sem relação temática real.

A complexidade O(n²) no número de sentenças limita a escalabilidade para textos muito longos. Em documentos com centenas de sentenças, o tempo de cálculo da matriz de similaridade cresce rapidamente.

### 11.3 Considerações Finais

Os três projetos desta série percorrem uma progressão natural na complexidade das abordagens de sumarização extrativa: da contagem simples de palavras, passando pela análise de clusters de proximidade, até a modelagem de grafos de similaridade. Cada algoritmo adiciona uma camada de sofisticação e resolve uma limitação do anterior, ao custo de maior complexidade de implementação e tempo de processamento. A escolha entre eles depende do tipo de texto, do volume de dados e do equilíbrio desejado entre qualidade do resumo e eficiência computacional.



---



> Material de estudo desenvolvido para o repositório [data-trivium](https://github.com/wellington-moreira-santos/data-trivium), a partir do curso de Sumarização de Textos com PLN da IAExpert Academy.  